# Exploratory Data Analysis: Student Performance Predictor

This notebook audits the modeling dataset before training. It checks missing values, duplicates, invalid ranges, outliers, univariate distributions, relationships between variables, correlations, and early feature importance signals.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline

from src.feature_engineering import add_engineered_features, get_model_feature_columns
from src.preprocessing import NUMERIC_COLUMNS, CATEGORICAL_COLUMNS, build_preprocessor, clean_student_data
from src.utils import load_or_create_dataset

sns.set_theme(style='whitegrid', palette='Set2')
pd.set_option('display.max_columns', 100)

## Load and Standardize Data

The project prefers the UCI Student Performance dataset and falls back to deterministic synthetic data when offline. Both paths are standardized into the same production schema.

In [ ]:
raw = load_or_create_dataset()
data = add_engineered_features(clean_student_data(raw))
data.head()

## Dataset Shape and Types

This table confirms the number of rows, feature types, and target availability. Numerical columns should represent measurable quantities, while categorical fields should contain compact domain values.

In [ ]:
print(f'Rows: {data.shape[0]:,}')
print(f'Columns: {data.shape[1]:,}')
data.info()

## Missing Values

The missing-value chart shows whether any feature requires imputation. Production preprocessing still includes imputers so inference remains robust if future records contain null values.

In [ ]:
missing = data.isna().sum().sort_values(ascending=False)
display(missing.to_frame('missing_count'))
missing[missing > 0].plot(kind='bar', figsize=(10, 4), title='Missing Values by Column')
plt.ylabel('Missing Count')
plt.tight_layout()
plt.show()

## Duplicate Records

Duplicate rows can inflate model confidence by allowing nearly identical examples to appear in train and test splits. The cleaning step removes exact duplicates before modeling.

In [ ]:
duplicate_count = data.duplicated().sum()
print(f'Duplicate rows: {duplicate_count}')

## Summary Statistics

The summary table checks center, spread, and extreme values. It is useful for catching impossible values such as negative attendance, grades above 100, or unrealistic sleep hours.

In [ ]:
data.describe().T

## Outlier Detection

The IQR table flags unusually high or low values. Outliers are not automatically removed because unusually low attendance or unusually high study time can be predictive, but the counts reveal where robust modeling choices matter.

In [ ]:
numeric = data.select_dtypes(include='number')
q1 = numeric.quantile(0.25)
q3 = numeric.quantile(0.75)
iqr = q3 - q1
outliers = ((numeric < (q1 - 1.5 * iqr)) | (numeric > (q3 + 1.5 * iqr))).sum()
outliers.sort_values(ascending=False).to_frame('iqr_outlier_count')

## Histograms

Histograms show the shape of each numerical feature. Skewed columns may benefit from tree-based models, while near-normal features work well with scaled linear models.

In [ ]:
data[NUMERIC_COLUMNS + ['final_score']].hist(figsize=(16, 12), bins=25, edgecolor='black')
plt.suptitle('Numerical Feature Histograms', y=1.02)
plt.tight_layout()
plt.show()

## Box Plots

Box plots visualize spread and outliers for each numerical feature. Attendance, assignments, and risk-related features often show meaningful tail behavior tied to student outcomes.

In [ ]:
plt.figure(figsize=(16, 8))
sns.boxplot(data=data[NUMERIC_COLUMNS + ['final_score']], orient='h')
plt.title('Box Plots for Numerical Features')
plt.tight_layout()
plt.show()

## Target Distribution

The final-score distribution shows the regression target shape. A balanced spread across low, medium, and high scores gives the model enough signal across the grading range.

In [ ]:
plt.figure(figsize=(9, 5))
sns.histplot(data=data, x='final_score', kde=True, bins=25)
plt.title('Final Score Distribution')
plt.xlabel('Final Score')
plt.tight_layout()
plt.show()

## Pass/Fail Distribution

The pass/fail count plot checks class balance. A heavily imbalanced target would require stronger class weighting, resampling, or threshold tuning.

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=data, x='pass_fail')
plt.title('Pass/Fail Class Distribution')
plt.xticks([0, 1], ['Fail', 'Pass'])
plt.tight_layout()
plt.show()

## Correlation Heatmap

The heatmap highlights linear relationships. Previous grade, study efficiency, attendance, assignments, and engagement should show positive relationships with final score, while risk index should move in the opposite direction.

In [ ]:
plt.figure(figsize=(14, 10))
corr = data.select_dtypes(include='number').corr()
sns.heatmap(corr, cmap='coolwarm', center=0, annot=False)
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()

## Pair Plot

The pair plot focuses on the most interpretable academic drivers. It reveals whether relationships with final score are linear, clustered, or separated by pass/fail status.

In [ ]:
pair_columns = ['final_score', 'previous_grade', 'study_hours', 'attendance', 'assignments_completed', 'academic_engagement_score', 'pass_fail']
sns.pairplot(data[pair_columns], hue='pass_fail', diag_kind='kde', corner=True)
plt.suptitle('Pair Plot of Academic Drivers', y=1.02)
plt.show()

## Categorical Distributions

These count plots show whether categorical variables contain enough representation across groups. Sparse categories are handled with one-hot encoding and unknown-category support.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, column in zip(axes.ravel(), CATEGORICAL_COLUMNS):
    sns.countplot(data=data, x=column, ax=ax)
    ax.set_title(f'Distribution of {column}')
    ax.tick_params(axis='x', rotation=30)
axes.ravel()[-1].axis('off')
plt.tight_layout()
plt.show()

## Feature Importance Preview

This chart trains a quick Random Forest regressor only for EDA. It is not the final model selection step; it helps identify which features are likely to influence score prediction.

In [ ]:
features = get_model_feature_columns(data)
x = data[features]
y = data['final_score']
importance_model = Pipeline([
    ('preprocessor', build_preprocessor(NUMERIC_COLUMNS, CATEGORICAL_COLUMNS)),
    ('model', RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)),
])
importance_model.fit(x, y)
names = importance_model.named_steps['preprocessor'].get_feature_names_out()
values = importance_model.named_steps['model'].feature_importances_
importance = pd.DataFrame({'feature': names, 'importance': values}).sort_values('importance', ascending=False).head(15)

plt.figure(figsize=(10, 7))
sns.barplot(data=importance, x='importance', y='feature')
plt.title('EDA Feature Importance Preview')
plt.tight_layout()
plt.show()
importance

## EDA Summary

The modeling pipeline should keep imputers for future missing values, remove exact duplicates, clip invalid ranges, scale numerical variables, one-hot encode categorical fields, and include engineered engagement/risk features. Model evaluation should include both regression and classification metrics because the application serves two different prediction tasks.